#Colab Mount and setup

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# After downloading the shared starting point folder as a Zip
# Unzip it and re-upload it to a location on your GDrive

# This command copies the contents from the folder you uploaded to GDrive, to the colab working dir
!cp -r /content/drive/MyDrive/'Projeto RI 2020'/ProjetoRI2020 /content

# Add working dir to the sys path, so that we can find the aux python files when running the Notebook
import sys
if not 'ProjetoRI2020' in sys.path:
  sys.path += ['/content/ProjetoRI2020']

# Finally install required dependencies to run the notebook
!pip install elasticsearch
!pip install bert-serving-client
!pip install transformers
!pip install spacy
!python -m spacy download en_core_web_sm

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/Projeto RI 2020/ProjetoRI2020': No such file or directory
✔ Download and installation successful
You can now load the model via spacy.load('en_core_web_sm')


#Imports

In [ ]:
import numpy as np
import pprint
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from transformers import BertTokenizer, BertTokenizerFast, BertModel
import torch
import TRECCASTeval as trec
import numpy as np
import pandas as pd
import ElasticSearchSimpleAPI as es
import pickle
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
import Graphics
import pprint
import spacy

#Preparar Bert Inputs para treino

In [ ]:
bert_model_name = 'bert-base-cased'
bert_model_name = 'nboost/pt-bert-base-uncased-msmarco'
CLS_token = "[CLS]"
SEP_token = "[SEP]"

elastic = es.ESSimpleAPI()
test_bed = trec.ConvSearchEvaluation()

triplets = [] #(utterance, passagem, relevance judgment) tuples
print(test_bed.relevance_judgments)
for index, row in test_bed.relevance_judgments.iterrows():

  if ('CAR' not in row['docid'] and 'WAPO' not in row['docid'] and 'MARCO_4286598' not in row['docid']):
    doc_body = elastic.get_doc_body(row['docid'])
    relev = row['rel']
    if(relev > 1):
      relev = 1
    turnId =  row['topic_turn_id'].split('_')
    triplets.append([utterances_entities_train[int(turnId[0])][int(turnId[1])],doc_body, relev])
    #triplets.append([test_bed.train_topics[int(turnId[0]) - 1]['turn'][int(turnId[1]) - 1]['raw_utterance'],doc_body, relev])

array = np.array(triplets)
df_triplets = pd.DataFrame(data = array, columns = ['topic_turn', 'passage', 'rel'])
df_triplets


tokenizer = BertTokenizerFast.from_pretrained(bert_model_name)
device = torch.device("cuda")
model = BertModel.from_pretrained('bert-base-uncased', return_dict=True)
model.eval()


# Receive a list of [query_text, doc_text] in variable sentences
# Returns a dictionary of tensors
# It can only receive a single pair [query_text, doc_text] each time
def convert_to_bert_input(sentences, max_seq_length=512, tokenizer=tokenizer, add_cls=True, padding='do_not_pad', truncation=False):
  return tokenizer.encode_plus(sentences, add_special_tokens=add_cls, padding=padding, max_length=max_seq_length, truncation=truncation,
                               return_tensors='pt', return_token_type_ids=True, return_attention_mask=True)



sentences = df_triplets[['topic_turn', 'passage']]
outputs = []
i = 0
for index, sentence in sentences.iterrows():
  inputs_qa = convert_to_bert_input(sentences=sentence, max_seq_length=512, tokenizer=tokenizer, padding='max_length', truncation=False)

  input_ids = inputs_qa['input_ids']
  attention_mask = inputs_qa['attention_mask']
  token_type_ids = inputs_qa['token_type_ids']

  input_ids = input_ids.to(device)
  attention_mask = attention_mask.to(device)
  token_type_ids = token_type_ids.to(device)

  outputs_qa = model(**inputs_qa)

  outputs_qa = outputs_qa["last_hidden_state"][0,0].detach().clone()

  outputs.append(outputs_qa.numpy())


     topic_turn_id  dummy          docid  rel
0              1_1      0   MARCO_955948    2
1              1_1      0  MARCO_6203672    2
2              1_1      0   MARCO_849267    0
3              1_1      0  MARCO_2331424    0
4              1_1      0  MARCO_4455128    0
...            ...    ...            ...  ...
2394          30_7      0  MARCO_4282236    0
2395          30_7      0  MARCO_6551052    0
2396          30_7      0  MARCO_4225659    0
2397          30_7      0  MARCO_6645461    0
2398          30_7      0  MARCO_4250016    0

[2399 rows x 4 columns]
['What are the educational requirements required to become one?', 'Physician assistants work under the supervision of a physician or surgeon; however, their specific duties and the extent to which they must be supervised differ from state to state. Physician assistants work in all areas of medicine, including primary care and family medicine, emergency medicine, and psychiatry.The work of physician assistants depends in

KeyboardInterrupt: ignored

In [ ]:
bert_model_name = 'bert-base-cased'
bert_model_name = 'nboost/pt-bert-base-uncased-msmarco'
CLS_token = "[CLS]"
SEP_token = "[SEP]"

elastic = es.ESSimpleAPI()
test_bed = trec.ConvSearchEvaluation()

triplets = [] #(utterance, passagem, relevance judgment) tuples
print(test_bed.relevance_judgments)
for index, row in test_bed.relevance_judgments.iterrows():

  if ('CAR' not in row['docid'] and 'WAPO' not in row['docid'] and 'MARCO_4286598' not in row['docid']):
    doc_body = elastic.get_doc_body(row['docid'])
    relev = row['rel']
    if(relev > 1):
      relev = 1
    turnId =  row['topic_turn_id'].split('_')
    triplets.append([utterances_joined_train[int(turnId[0])][int(turnId[1])],doc_body, relev])
    #triplets.append([test_bed.train_topics[int(turnId[0]) - 1]['turn'][int(turnId[1]) - 1]['raw_utterance'],doc_body, relev])

array = np.array(triplets)
df_triplets = pd.DataFrame(data = array, columns = ['topic_turn', 'passage', 'rel'])
df_triplets


tokenizer = BertTokenizerFast.from_pretrained(bert_model_name)
device = torch.device("cuda")
model = BertModel.from_pretrained('bert-base-uncased', return_dict=True)
model.eval()


# Receive a list of [query_text, doc_text] in variable sentences
# Returns a dictionary of tensors
# It can only receive a single pair [query_text, doc_text] each time
def convert_to_bert_input(sentences, max_seq_length=512, tokenizer=tokenizer, add_cls=True, padding='do_not_pad', truncation=False):
  return tokenizer.encode_plus(sentences, add_special_tokens=add_cls, padding=padding, max_length=max_seq_length, truncation=truncation,
                               return_tensors='pt', return_token_type_ids=True, return_attention_mask=True)



sentences = df_triplets[['topic_turn', 'passage']]
outputs = []
i = 0
for index, sentence in sentences.iterrows():
  inputs_qa = convert_to_bert_input(sentences=sentence, max_seq_length=512, tokenizer=tokenizer, padding='max_length', truncation=False)

  input_ids = inputs_qa['input_ids']
  attention_mask = inputs_qa['attention_mask']
  token_type_ids = inputs_qa['token_type_ids']

  input_ids = input_ids.to(device)
  attention_mask = attention_mask.to(device)
  token_type_ids = token_type_ids.to(device)

  outputs_qa = model(**inputs_qa)

  outputs_qa = outputs_qa["last_hidden_state"][0,0].detach().clone()

  outputs.append(outputs_qa.numpy())

Seleção dos melhores parâmetros e treino com a Regressão Logística

In [ ]:
logistic = LogisticRegression()

# Create regularization penalty space
penalty = ['l1', 'l2']
max_iter = [100, 200, 300, 400, 500]


# Create regularization hyperparameter space
C = [0.01*(2**i) for i in range(0,10)]

tol = [0.01*(2**i) for i in range(0,10)]

# Create hyperparameter options
hyperparameters = dict(C=C, penalty=penalty, tol = tol, max_iter = max_iter)

clf = GridSearchCV(logistic, hyperparameters, cv=5, verbose=0)
xs = outputs
ys = df_triplets['rel']
clf.fit(xs, ys)

clf.best_estimator_.get_params()

Identificar entidades

In [ ]:
elastic = es.ESSimpleAPI()
test_bed = trec.ConvSearchEvaluation()

Juntar entities

In [ ]:
nlp = spacy.load('en_core_web_sm')
utterances_entities_test = {}
for topic in test_bed.test_topics:
  utterances_entities_test[topic['number']] = []
  entities = []
  for turn in topic['turn']:
    entities.append(nlp(turn['raw_utterance']).ents)
    utterance = turn['raw_utterance']
    for entitie in entities:
      if len(entitie):
        utterance = ' '.join([utterance, str(entitie[0])])
    utterances_entities_test[topic['number']].append(utterance)

pprint.pprint(utterances_entities_test)



{31: ['What is throat cancer?',
      'Is it treatable?',
      'Tell me about lung cancer.',
      'What are its symptoms? ',
      'Can it spread to the throat?',
      'What causes throat cancer?',
      'What is the first sign of it? first',
      'Is it the same as esophageal cancer? first',
      "What's the difference in their symptoms? first"],
 32: ['What are the different types of sharks?',
      'Are sharks endangered?  If so, which species?',
      'Tell me more about tiger sharks.',
      'What is the largest ever to have lived on Earth? Earth',
      "What's the biggest ever caught? Earth",
      'What about for great whites? Earth',
      'Tell me about makos. Earth',
      'What are their adaptations? Earth',
      'Where do they live? Earth',
      'What do they eat? Earth',
      'How do they compare with tigers for being dangerous? Earth'],
 33: ['Tell me about the Neverending Story film. the Neverending Story',
      'What is it about? the Neverending Story',
      

In [ ]:
nlp = spacy.load('en_core_web_sm')
utterances_entities_train = {}
for topic in test_bed.train_topics:
  utterances_entities_train[topic['number']] = []
  entities = []
  for turn in topic['turn']:
    entities.append(nlp(turn['raw_utterance']).ents)
    utterance = turn['raw_utterance']
    for entitie in entities:
      if len(entitie):
        utterance = ' '.join([utterance, str(entitie[0])])
    utterances_entities_train[topic['number']].append(utterance)

Juntar turns

In [ ]:
utterances_joined_test = {}
for topic in test_bed.test_topics:
  #topic = test_bed.test_topics[0]
  utterances_joined_test[topic['number']] = []
  utterances = ''
  for turn in topic['turn']:
    utterances = ' '.join([turn['raw_utterance'], utterances])
    utterances_joined_test[topic['number']].append(utterances)

pprint.pprint(utterances_joined_test)

{31: ['What is throat cancer? ',
      'Is it treatable? What is throat cancer? ',
      'Tell me about lung cancer. Is it treatable? What is throat cancer? ',
      'What are its symptoms?  Tell me about lung cancer. Is it treatable? '
      'What is throat cancer? ',
      'Can it spread to the throat? What are its symptoms?  Tell me about lung '
      'cancer. Is it treatable? What is throat cancer? ',
      'What causes throat cancer? Can it spread to the throat? What are its '
      'symptoms?  Tell me about lung cancer. Is it treatable? What is throat '
      'cancer? ',
      'What is the first sign of it? What causes throat cancer? Can it spread '
      'to the throat? What are its symptoms?  Tell me about lung cancer. Is it '
      'treatable? What is throat cancer? ',
      'Is it the same as esophageal cancer? What is the first sign of it? What '
      'causes throat cancer? Can it spread to the throat? What are its '
      'symptoms?  Tell me about lung cancer. Is it treata

In [ ]:
utterances_joined_train = {}
for topic in test_bed.train_topics:
  #topic = test_bed.test_topics[0]
  utterances_joined_train[topic['number']] = []
  utterances = ''
  for turn in topic['turn']:
    utterances = ' '.join([turn['raw_utterance'], utterances])
    utterances_joined_train[topic['number']].append(utterances)

pprint.pprint(utterances_joined_train)

{1: ["What is a physician's assistant? ",
     'What are the educational requirements required to become one? What is a '
     "physician's assistant? ",
     'What does it cost? What are the educational requirements required to '
     "become one? What is a physician's assistant? ",
     "What's the average starting salary in the UK? What does it cost? What "
     'are the educational requirements required to become one? What is a '
     "physician's assistant? ",
     "What about in the US? What's the average starting salary in the UK? What "
     'does it cost? What are the educational requirements required to become '
     "one? What is a physician's assistant? ",
     'What school subjects are needed to become a registered nurse? What about '
     "in the US? What's the average starting salary in the UK? What does it "
     'cost? What are the educational requirements required to become one? What '
     "is a physician's assistant? ",
     'What is the PA average salary vs an RN? 

Relevancias com LMD

In [ ]:
_recall = 0
_p10 = 0
_ndcg5 = 0
_nturns = 0
for topic in test_bed.test_topics:

    conv_id = topic['number']
    if conv_id not in (31, 32, 33, 34, 37, 40, 49, 50, 54, 56, 58, 59, 61, 67, 68, 69, 75, 77, 78, 79):
        continue

    for turn in topic['turn']:
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)

        aux = test_bed.test_relevance_judgments.loc[test_bed.test_relevance_judgments['topic_turn_id'] == (topic_turn_id)]
        num_rel = aux.loc[aux['rel'] != 0]['docid'].count()

        if num_rel == 0:
            continue

        result = elastic.search_body(query=utterance, numDocs = 100)
        if np.size(result) == 0 or num_rel == 0:
            print(topic_turn_id, utterance, num_rel, "NO RESULTS")
            continue
        else:
            print(topic_turn_id, utterance, num_rel)

        [p10, recall, ap, ndcg5], recall_var, precision_var = test_bed.eval(result[['_id','_score']], topic_turn_id)

#        print('P10=', p10, '  Recall=', recall, '  NDCG=',ndcg)
        _recall = _recall + recall
        _p10 = _p10 + p10
        _ndcg5 = _ndcg5 + ndcg5

        _nturns = _nturns + 1

_p10 = _p10/_nturns
_recall = _recall/_nturns
_ndcg5 = _ndcg5/_nturns

print()
print('P10=', _p10, '  Recall=', _recall, '  NDCG@5', _ndcg5)


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


31_1 What is throat cancer? 89


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


31_2 Is it treatable? 77


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


...

/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


79_5 How is his work related to Comte? 41
79_6 What is the functionalist theory? 77


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


79_7 What is its main criticism? 23
79_8 How does it compare to conflict theory? 51
79_9 What are modern examples of conflict theory? 40

P10= 0.23895348837209304   Recall= 0.16716739361136979   NDCG@5 0.18025607112012662


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


Relevancias com LMD - entities

In [ ]:
_recall = 0
_p10 = 0
_ndcg5 = 0
_nturns = 0
for topic in test_bed.test_topics:

    conv_id = topic['number']
    if conv_id not in (31, 32, 33, 34, 37, 40, 49, 50, 54, 56, 58, 59, 61, 67, 68, 69, 75, 77, 78, 79):
        continue

    for i in range(len(topic['turn'])):
        turn = topic['turn'][i]
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)

        aux = test_bed.test_relevance_judgments.loc[test_bed.test_relevance_judgments['topic_turn_id'] == (topic_turn_id)]
        num_rel = aux.loc[aux['rel'] != 0]['docid'].count()

        if num_rel == 0:
            continue
        query = utterances_entities[conv_id][i]
        print(query)
        result = elastic.search_body(query, numDocs = 100)
        if np.size(result) == 0 or num_rel == 0:
            continue

        [p10, recall, ap, ndcg5], recall_var, precision_var = test_bed.eval(result[['_id','_score']], topic_turn_id)

#        print('P10=', p10, '  Recall=', recall, '  NDCG=',ndcg)
        _recall = _recall + recall
        _p10 = _p10 + p10
        _ndcg5 = _ndcg5 + ndcg5

        _nturns = _nturns + 1

_p10 = _p10/_nturns
_recall = _recall/_nturns
_ndcg5 = _ndcg5/_nturns

print()
print('P10=', _p10, '  Recall=', _recall, '  NDCG@5', _ndcg5)


What is throat cancer?


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


Is it treatable?


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


Tell me about lung cancer.
What are its symptoms? 


...

/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


What is the functionalist theory? Auguste Comte Herbert Spencer Comte
What is its main criticism? Auguste Comte Herbert Spencer Comte


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


How does it compare to conflict theory? Auguste Comte Herbert Spencer Comte
What are modern examples of conflict theory? Auguste Comte Herbert Spencer Comte

P10= 0.3040462427745665   Recall= 0.20460828352134908   NDCG@5 0.2571048440341999


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


Relevancias com LMD - utterances

In [ ]:
_recall = 0
_p10 = 0
_ndcg5 = 0
_nturns = 0
for topic in test_bed.test_topics:

    conv_id = topic['number']
    if conv_id not in (31, 32, 33, 34, 37, 40, 49, 50, 54, 56, 58, 59, 61, 67, 68, 69, 75, 77, 78, 79):
        continue

    for i in range(len(topic['turn'])):
        turn = topic['turn'][i]
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)

        aux = test_bed.test_relevance_judgments.loc[test_bed.test_relevance_judgments['topic_turn_id'] == (topic_turn_id)]
        num_rel = aux.loc[aux['rel'] != 0]['docid'].count()

        if num_rel == 0:
            continue
        query = utterances_entities[conv_id][i]
        print(query)
        result = elastic.search_body(query=utterance, numDocs = 100)

        if np.size(result) == 0 or num_rel == 0:
            print(topic_turn_id, utterance, num_rel, "NO RESULTS")
            continue
        else:
            print(topic_turn_id, utterance, num_rel)

        [p10, recall, ap, ndcg5], recall_var, precision_var  = test_bed.eval(result[['_id','_score']], topic_turn_id)

#        print('P10=', p10, '  Recall=', recall, '  NDCG=',ndcg)
        _recall = _recall + recall
        _p10 = _p10 + p10
        _ndcg5 = _ndcg5 + ndcg5

        _nturns = _nturns + 1

_p10 = _p10/_nturns
_recall = _recall/_nturns
_ndcg5 = _ndcg5/_nturns

print()
print('P10=', _p10, '  Recall=', _recall, '  NDCG@5', _ndcg5)


What is throat cancer?


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


31_1 What is throat cancer? 89
Is it treatable?


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


31_2 Is it treatable? 77
Tell me about lung cancer.


...

/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


79_5 How is his work related to Comte? 41
What is the functionalist theory? Auguste Comte Herbert Spencer Comte
79_6 What is the functionalist theory? 77
What is its main criticism? Auguste Comte Herbert Spencer Comte
79_7 What is its main criticism? 23
How does it compare to conflict theory? Auguste Comte Herbert Spencer Comte


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])
/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


79_8 How does it compare to conflict theory? 51
What are modern examples of conflict theory? Auguste Comte Herbert Spencer Comte
79_9 What are modern examples of conflict theory? 40

P10= 0.23895348837209304   Recall= 0.16716739361136979   NDCG@5 0.18025607112012662


/content/ProjectoRI2020/ElasticSearchSimpleAPI.py:18: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead
  df = json_normalize(result["hits"]["hits"])


#The T5 rewriting class that you should copy to one of your files

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Copy the files and the data to the Colab Workspace
!rm -rf /content/ProjectoRI2020
#!cp -r /content/drive/MyDrive/Work/PhD/TAships/RI2020/ProjectoRI2020 /content
!cp -r /content/drive/MyDrive/*COMPLETE WITH THE LOCATION OF THE SHARED FOLDER ON YOUR DRIVE* /content

# Add project dir to filepath so that our notebook can find the provided python files
import sys
if not 'ProjectoRI2020' in sys.path:
  sys.path += ['/content/ProjectoRI2020']

# Install required packages to run the code. You only need to run this once on your local machine.
# However, in colab whenever you login you are given a new machine and working enviroment, thus we need to do it everytime we log in
!pip install elasticsearch

KeyboardInterrupt: ignored

In [ ]:
%tensorflow_version 2.x
!pip install t5==0.5.0
import tensorflow as tf
import tensorflow_text

     |████████████████████████████████| 143kB 6.7MB/s 
     |████████████████████████████████| 1.1MB 24.7MB/s 
     |████████████████████████████████| 3.7MB 33.4MB/s 
     |████████████████████████████████| 71kB 7.7MB/s 
     |████████████████████████████████| 2.6MB 34.2MB/s 
     |████████████████████████████████| 348kB 35.2MB/s 
     |████████████████████████████████| 1.4MB 39.1MB/s 
     |████████████████████████████████| 890kB 35.6MB/s 
     |████████████████████████████████| 2.9MB 37.4MB/s 
  Created wheel for sacremoses: filename=sacremoses-0.0.43-cp36-none-any.whl size=893257 sha256=342efc724b0bddd2cc352660da44af7b8269a169f8b671a2dd4a13ddc7736419
  Stored in directory: /root/.cache/pip/wheels/29/3c/fd/7ce5c3f0666dab31a50123635e6fb5e19ceb42ce38d4e58f45
Successfully built sacremoses


In [ ]:
!rm -rf /content/t5-canard
# !cp -r /content/drive/MyDrive/Work/PhD/TAships/RI2020/t5-canard.zip/ /content/t5-canard.zip
!cp -r /content/drive/' ' /content/t5-canard.zip
!unzip /content/t5-canard.zip

In [ ]:

class QueryRewriterT5:
  def __init__(self, model_path="/content/t5-canard"):
    """
      Loads T5 model for prediction
      Returns the model
    """
    if tf.executing_eagerly():
        print("Loading SavedModel in eager mode.")
        imported = tf.saved_model.load(model_path, ["serve"])
        self.t5_model = lambda x: imported.signatures['serving_default'](tf.constant(x))['outputs'].numpy()
    else:
        print("Loading SavedModel in tf 1.x graph mode.")
        tf.compat.v1.reset_default_graph()
        sess = tf.compat.v1.Session()
        meta_graph_def = tf.compat.v1.saved_model.load(sess, ["serve"], model_path)
        signature_def = meta_graph_def.signature_def["serving_default"]
        self.t5_model = lambda x: sess.run(
            fetches=signature_def.outputs["outputs"].name,
            feed_dict={signature_def.inputs["input"].name: x}
        )

  """
    query: str - the query string to be rewritten using T5
    ctx_list: list - A list of strings containing the turns or text to give context to T5
    Returns a string with the rewritten query
  """
  def rewrite_query_with_T5(self, _curr_query, _ctx_list):
    _t5_query = '{} [CTX] '.format(_curr_query) + ' [TURN] '.join(_ctx_list)
    print("Query and context: {}".format(_t5_query))
    return self.t5_model([_t5_query])[0].decode('utf-8')

  """
    queries_list: list - A list of strings containing the raw utterances ordered from first to last
    Returns a list of strings with the rewritten queries
  """
  def rewrite_dialog_with_T5(self, _queries_list):
    _rewritten_queries_list=[]
    for i in range(len(_queries_list)):
      _current_query = _queries_list[i]
      _rewritten_query = self.rewrite_query_with_T5(_current_query, _queries_list[:i])
      print("Rewritten query: {}\n".format(_rewritten_query))
      _rewritten_queries_list.append(_rewritten_query)
    return _rewritten_queries_list